In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import numpy as np
import cv2
import matplotlib.pyplot as plt

# ============================================================
# 1. SETTINGS
# ============================================================
train_dir = r"C:\Users\Sharooz Ali\Desktop\Local Disk(D)\Python_MachineLearning\asl_alphabet_train\asl_alphabet_train"

IMG_SIZE   = 96
BATCH_SIZE = 32

# ============================================================
# 2. DATA GENERATORS — FIX: Alag alag datagen banaye
# ============================================================

# Training ke liye — augmentation ON
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.15,
    rotation_range=15,
    zoom_range=0.15,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest'
)

# ✅ FIX: Validation ke liye SIRF rescale — koi augmentation nahi
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.15
)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(   # ✅ alag datagen
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

num_classes = train_generator.num_classes
print(f"Total Classes: {num_classes}")
print(f"Class Names  : {list(train_generator.class_indices.keys())}")

# ============================================================
# 3. MODEL
# ============================================================
model = Sequential([
    # Block 1
    Conv2D(32, (3,3), activation='relu', padding='same',
           input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    BatchNormalization(),
    Conv2D(32, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Dropout(0.25),

    # Block 2
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Dropout(0.25),

    # Block 3
    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Dropout(0.25),

    # Block 4
    Conv2D(256, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Dropout(0.25),

    # Head
    tf.keras.layers.GlobalAveragePooling2D(),
    Dense(512, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(256, activation='relu'),
    Dropout(0.4),
    Dense(num_classes, activation='softmax')
])

model.summary()

# ============================================================
# 4. COMPILE
# ============================================================
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ============================================================
# 5. CALLBACKS
# ============================================================
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "asl_best_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

# ============================================================
# 6. TRAIN
# ============================================================
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,                        # zyada epochs — early stop handle karega
    callbacks=[early_stop, reduce_lr, checkpoint]
)

# ============================================================
# 7. TRAINING GRAPHS
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'],     label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['loss'],     label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.savefig('training_history.png')
plt.show()

# ============================================================
# 8. PREDICT FUNCTION — ✅ FIX: Sahi preprocess ke saath
# ============================================================
class_names = list(train_generator.class_indices.keys())

def predict_image(img_path):
    """
    Koi bhi image do — sahi se predict karega.
    Google se liya ho ya khud liya ho.
    """
    # Image padhna
    img = cv2.imread(img_path)
    if img is None:
        print("❌ Image nahi mili — path check karo!")
        return

    # ✅ BGR to RGB (OpenCV BGR mein padhta hai, model RGB chahta hai)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # ✅ Same size jaise training mein tha
    img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))

    # ✅ Same normalize jaise training mein tha (rescale=1./255)
    img_normalized = img_resized / 255.0

    # ✅ Batch dimension add karo — model (1, 96, 96, 3) chahta hai
    img_batch = np.expand_dims(img_normalized, axis=0)

    # Predict
    predictions = model.predict(img_batch, verbose=0)
    class_idx   = np.argmax(predictions)
    confidence  = np.max(predictions) * 100

    # Top 3 results
    top3_idx  = np.argsort(predictions[0])[::-1][:3]
    top3      = [(class_names[i], predictions[0][i]*100) for i in top3_idx]

    print(f"\n{'='*40}")
    print(f"Image     : {img_path}")
    print(f"Prediction: {class_names[class_idx]}")
    print(f"Confidence: {confidence:.2f}%")
    print(f"\nTop 3 Results:")
    for rank, (name, conf) in enumerate(top3, 1):
        print(f"  {rank}. {name:10s} — {conf:.2f}%")
    print('='*40)

    # Show image with prediction
    plt.figure(figsize=(4, 4))
    plt.imshow(img_rgb)
    plt.title(f"Predicted: {class_names[class_idx]} ({confidence:.1f}%)")
    plt.axis('off')
    plt.show()

    return class_names[class_idx], confidence


# ============================================================
# 9. MEDIAPIPE — Google Images ke liye Hand Crop
# ============================================================
def predict_with_hand_crop(img_path):
    """
    Google images ke liye — pehle haath crop karo, phir predict karo.
    pip install mediapipe
    """
    try:
        import mediapipe as mp

        mp_hands = mp.solutions.hands
        img      = cv2.imread(img_path)
        img_rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        with mp_hands.Hands(static_image_mode=True, max_num_hands=1) as hands:
            results = hands.process(img_rgb)

            if results.multi_hand_landmarks:
                h, w    = img.shape[:2]
                hand_lm = results.multi_hand_landmarks[0]

                x_list = [lm.x * w for lm in hand_lm.landmark]
                y_list = [lm.y * h for lm in hand_lm.landmark]

                # Thoda margin rakhna
                margin = 30
                x1 = max(0, int(min(x_list)) - margin)
                x2 = min(w, int(max(x_list)) + margin)
                y1 = max(0, int(min(y_list)) - margin)
                y2 = min(h, int(max(y_list)) + margin)

                hand_crop = img[y1:y2, x1:x2]

                # Cropped image save karo
                crop_path = img_path.replace('.jpg', '_cropped.jpg')
                cv2.imwrite(crop_path, hand_crop)
                print(f"✅ Hand crop ho gaya: {crop_path}")

                # Ab predict karo
                return predict_image(crop_path)
            else:
                print("⚠️ Hand detect nahi hua — direct predict karta hoon...")
                return predict_image(img_path)

    except ImportError:
        print("MediaPipe install nahi hai — 'pip install mediapipe' chalao")
        print("Direct predict kar raha hoon...")
        return predict_image(img_path)


# ============================================================
# 10. USE KARNE KA TARIKA
# ============================================================

# Normal image predict karo:
# predict_image(r"C:\path\to\your\image.jpg")

# Google se liya image predict karo (hand crop ke saath):
# predict_with_hand_crop(r"C:\path\to\google_image.jpg")

In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import numpy as np
import cv2
import matplotlib.pyplot as plt

In [3]:
train_dir = r"C:\Users\Sharooz Ali\Desktop\Local Disk(D)\Python_MachineLearning\asl_alphabet_train\asl_alphabet_train"

In [3]:
IMG_SIZE   = 96
BATCH_SIZE = 32

In [4]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.15,
    rotation_range=15,
    zoom_range=0.15,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest'
)

In [5]:
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.15
)


In [6]:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

Found 42013 images belonging to 26 classes.


In [ ]:
val_generator = val_datagen.flow_from_directory(  
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

Found 7404 images belonging to 26 classes.


In [8]:
num_classes = train_generator.num_classes
print(f"Total Classes: {num_classes}")
print(f"Class Names  : {list(train_generator.class_indices.keys())}")

Total Classes: 26
Class Names  : ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']


In [9]:
model = Sequential([
    # Block 1
    Conv2D(32, (3,3), activation='relu', padding='same',
           input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    BatchNormalization(),
    Conv2D(32, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Dropout(0.25),

    # Block 2
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Dropout(0.25),

    # Block 3
    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Dropout(0.25),

    # Block 4
    Conv2D(256, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    Dropout(0.25),

    # Head
    tf.keras.layers.GlobalAveragePooling2D(),
    Dense(512, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(256, activation='relu'),
    Dropout(0.4),
    Dense(num_classes, activation='softmax')
])


C:\Users\Sharooz Ali\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [11]:
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "asl_best_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)


In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,                        # zyada epochs — early stop handle karega
    callbacks=[early_stop, reduce_lr, checkpoint]
)

Epoch 1/30


C:\Users\Sharooz Ali\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1313/1313 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.1987 - loss: 3.0139
Epoch 1: val_accuracy improved from -inf to 0.39789, saving model to asl_best_model.keras
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 1464s 1s/step - accuracy: 0.1989 - loss: 3.0132 - val_accuracy: 0.3979 - val_loss: 2.0145 - learning_rate: 5.0000e-04
Epoch 2/30
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 0s 968ms/step - accuracy: 0.8142 - loss: 0.5465
Epoch 2: val_accuracy improved from 0.39789 to 0.89803, saving model to asl_best_model.keras
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 1319s 1s/step - accuracy: 0.8143 - loss: 0.5464 - val_accuracy: 0.8980 - val_loss: 0.4998 - learning_rate: 5.0000e-04
Epoch 3/30
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 0s 982ms/step - accuracy: 0.9325 - loss: 0.2101
Epoch 3: val_accuracy improved from 0.89803 to 0.90411, saving model to asl_best_model.keras
1313/1313 ━━━━━━━━━━━━━━━━━━━━ 1338s 1s/step - accuracy: 0.9325 - loss: 0.2100 - val_accuracy: 0.9041 - val_loss: 0.4528 - learning_rate: 5.0000e-04
Epoch 4/30
1313/1313 